# Vector Store Utility Reference

Developer-facing statements defined in `langchain_core.vectorstores.utils`.

These utilities are marked as private API in the pinned source and may change without notice.

# `Matrix`

Type-only alias for matrix inputs accepted by internal vector-store utilities.

```python
Matrix = (
    list[list[float]]
    | list[npt.NDArray[np.floating]]
    | npt.NDArray[np.floating]
)
```

The alias is defined only when `TYPE_CHECKING` is true and is not available as a runtime module attribute.

---

# `maximal_marginal_relevance`

Selects embedding indices using maximal marginal relevance, balancing query similarity against redundancy among already selected embeddings.

```python
maximal_marginal_relevance(
    query_embedding: npt.NDArray[np.floating], # Query embedding
    embedding_list: list[list[float]], # Candidate embeddings
    lambda_mult: float = 0.5, # Balance between query relevance and diversity
    k: int = 4, # Maximum number of indices to select
) -> list[int] # Selected candidate indices
```

A one-dimensional query embedding is expanded into a single-row matrix. The most query-similar candidate is selected first, followed by candidates maximizing the relevance-diversity objective.

Returns an empty list when `k` is non-positive or `embedding_list` is empty. When `k` exceeds the number of candidates, all candidate indices are selected.

Raises `ImportError` when NumPy is unavailable. Errors and warnings produced while calculating cosine similarity may also propagate, including `ValueError` for incompatible embedding dimensions or unusable all-NaN similarity values, and `RuntimeWarning` for NaN or infinite inputs.

In [ ]:
#%pip install -U langchain-core numpy # Install the required libraries

import numpy as np # Import NumPy for creating embedding arrays
from langchain_core.vectorstores.utils import maximal_marginal_relevance # Import the MMR utility function


documents = [ # Create sample documents
    "Python is used for machine learning.", # Create the first Python document
    "Python is popular for data science.", # Create another similar Python document
    "Vector databases store embedding vectors.", # Create a vector-database document
    "LangChain helps build retrieval applications.", # Create a LangChain document
    "Machine learning models learn from data.", # Create a machine-learning document
] # Finish the document list


query_embedding = np.array( # Create the query embedding
    [1.0, 0.8, 0.1, 0.0], # Represent interest in Python and machine learning
    dtype=float, # Store the vector values as floating-point numbers
) # Finish creating the query embedding


document_embeddings = [ # Create embeddings for all candidate documents
    [1.0, 0.9, 0.0, 0.0], # Represent the first Python document
    [0.9, 0.8, 0.0, 0.0], # Represent the second similar Python document
    [0.0, 0.1, 1.0, 0.8], # Represent the vector-database document
    [0.1, 0.0, 0.8, 1.0], # Represent the LangChain retrieval document
    [0.5, 1.0, 0.1, 0.0], # Represent the machine-learning document
] # Finish the candidate embedding list


selected_indices = maximal_marginal_relevance( # Select relevant and diverse document indices
    query_embedding=query_embedding, # Supply the query embedding
    embedding_list=document_embeddings, # Supply all candidate embeddings
    lambda_mult=0.5, # Balance relevance and diversity equally
    k=3, # Select three documents
) # Finish the MMR selection


selected_documents = [documents[index] for index in selected_indices] # Convert selected indices into documents


print(f"Selected indices: {selected_indices}") # Display the selected candidate indices
print("\nSelected documents:") # Display the selected-document heading


for position, document in enumerate(selected_documents, start=1): # Process every selected document
    print(f"{position}. {document}") # Display the selected document

In [ ]:
relevance_indices = maximal_marginal_relevance( # Select documents with maximum relevance preference
    query_embedding=query_embedding, # Supply the query embedding
    embedding_list=document_embeddings, # Supply all candidate embeddings
    lambda_mult=1.0, # Prioritize query relevance completely
    k=3, # Select three documents
) # Finish relevance-focused selection


diversity_indices = maximal_marginal_relevance( # Select documents with maximum diversity preference
    query_embedding=query_embedding, # Supply the query embedding
    embedding_list=document_embeddings, # Supply all candidate embeddings
    lambda_mult=0.0, # Prioritize diversity completely after the first result
    k=3, # Select three documents
) # Finish diversity-focused selection


print("\nRelevance-focused results:") # Display the relevance-result heading


for index in relevance_indices: # Process every relevance-focused index
    print(f"{index}: {documents[index]}") # Display the selected document


print("\nDiversity-focused results:") # Display the diversity-result heading


for index in diversity_indices: # Process every diversity-focused index
    print(f"{index}: {documents[index]}") # Display the selected document